# 06 - Sincronizar metadata (Lambda + backfill) a RDS

## Objetivo
La Lambda creada anteriormente (por cada archivo nuevo que llega a `raw_data/`) y nuestro backfill (para los archivos que ya existían) escriben un JSON pequeño por archivo en `s3://xideralaws-curso-proyecto-alan/metadata/year=.../month=.../...json`. Este notebook lee todos esos JSON y los sincroniza a la tabla `pipeline_metadata` en RDS.

**Kernel: Python 3** — son puros archivos JSON pequeños, no hace falta Spark.

**Por qué estamos sincronizando y no reemplazando como en el notebook 05 (exportar-rds):** a diferencia de los KPIs de Gold (que se recalculan completos en cada ejecución), la metadata se va acumulando archivo por archivo con el tiempo, ya que cada vez que llega un parquet nuevo, la Lambda agrega un JSON más. Por eso aquí usamos `INSERT ... ON DUPLICATE KEY UPDATE` en vez de `to_sql(replace)`: si el `s3_key` ya existe en la tabla, actualiza esa fila; si no existe, la inserta. Así podemos ejecutar este notebook las veces que queramos sin duplicar filas ni perder las que ya estaban; esto nos brinda idempotencia, igual que la Lambda y el backfill.

## Prerrequisito
Debemos haber ya creado la tabla pipeline_metadata; para esto ejecutamos el comando `CREATE TABLE pipeline_metadata` (en mi caso puntual, yo creé la tabla usando el gestor de bases de datos DBeaver), ya que es primordial tener nuestra tabla pipeline_metadata lista, ya que propiamente este notebook no crea la tabla, solo la llena.

In [1]:
import os
import getpass
import json
from datetime import datetime, timezone
import boto3
import pandas as pd
from sqlalchemy import create_engine, text

BUCKET = "xideralaws-curso-proyecto-alan"
PREFIX = "metadata/"

mysql_host = os.getenv("MYSQL_HOST") or input("Endpoint de RDS MySQL: ")
mysql_port = int(os.getenv("MYSQL_PORT", "3306"))
mysql_user = os.getenv("MYSQL_USER") or input("Usuario de MySQL: ")
mysql_db = os.getenv("MYSQL_DB") or input("Base de datos (schema): ")
mysql_password = os.getenv("MYSQL_PASSWORD") or getpass.getpass("Contraseña de MySQL: ")

connection_string = f"mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{mysql_db}"
engine = create_engine(connection_string)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

s3_client = boto3.client("s3", region_name="us-west-1")
print("Conexion a RDS exitosa, base de datos:", mysql_db)

Endpoint de RDS MySQL:  database-1.cd8w4cuu6a79.us-west-1.rds.amazonaws.com
Usuario de MySQL:  admin
Base de datos (schema):  proyecto_integrador_alan
Contraseña de MySQL:  ········


Conexion a RDS exitosa, base de datos: proyecto_integrador_alan


### Listar todos los JSON de metadata en S3
Usamos el paginador porque puede haber más de 1000 objetos (el límite por página de S3).

In [2]:
paginator = s3_client.get_paginator("list_objects_v2")
claves_metadata = []

for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".json"):
            claves_metadata.append(obj["Key"])

print(f"Archivos de metadata encontrados en S3: {len(claves_metadata)}")

Archivos de metadata encontrados en S3: 117


### Función para convertir la fecha ISO del JSON a un formato que MySQL entienda
Lambda y el backfill guardan `registered_at` como `datetime.now(timezone.utc).isoformat()` (con zona horaria incluida). MySQL `DATETIME` no guarda zona horaria, así que la convertimos a UTC "naive" antes de insertarla.

In [3]:
def parsear_fecha(iso_str):
    dt = datetime.fromisoformat(iso_str)
    return dt.astimezone(timezone.utc).replace(tzinfo=None)

upsert_sql = text("""
    INSERT INTO pipeline_metadata
        (s3_bucket, s3_key, taxi_type, file_year, file_month, file_size_bytes, num_rows, registered_by, registered_at)
    VALUES
        (:s3_bucket, :s3_key, :taxi_type, :file_year, :file_month, :file_size_bytes, :num_rows, :registered_by, :registered_at)
    ON DUPLICATE KEY UPDATE
        file_size_bytes = VALUES(file_size_bytes),
        num_rows = VALUES(num_rows),
        registered_by = VALUES(registered_by),
        registered_at = VALUES(registered_at)
""")

print("Funcion y statement de upsert listos")

Funcion y statement de upsert listos


### Sincronizar: leer cada JSON y hacer upsert en RDS
Como son cientos de archivos pequeños (no millones de filas), un `for` normal es suficiente; no es necesario aplicar paralelismo.

In [4]:
sincronizados = 0
errores = []

with engine.begin() as conn:
    for clave in claves_metadata:
        try:
            respuesta = s3_client.get_object(Bucket=BUCKET, Key=clave)
            payload = json.loads(respuesta["Body"].read())

            conn.execute(upsert_sql, {
                "s3_bucket": payload["s3_bucket"],
                "s3_key": payload["s3_key"],
                "taxi_type": payload["taxi_type"],
                "file_year": payload["file_year"],
                "file_month": payload["file_month"],
                "file_size_bytes": payload.get("file_size_bytes"),
                "num_rows": payload.get("num_rows"),
                "registered_by": payload.get("registered_by"),
                "registered_at": parsear_fecha(payload["registered_at"]),
            })
            sincronizados += 1

        except Exception as error:
            errores.append((clave, str(error)))

print(f"Sincronizados correctamente: {sincronizados}/{len(claves_metadata)}")
if errores:
    print(f"\nArchivos con error ({len(errores)}):")
    for clave, detalle in errores[:10]:
        print(f"  - {clave}: {detalle}")

Sincronizados correctamente: 117/117


### Verificación
Comparamos el total en RDS contra el total de archivos que encontramos en S3, es crucial que coincidan (o ser mayor en RDS si ya había una sincronización previa con archivos que luego se borraron de S3, lo cual no aplica en este caso, pero lo menciono por si llega a darse la situación).

In [5]:
with engine.connect() as conn:
    total_rds = conn.execute(text("SELECT COUNT(*) FROM pipeline_metadata")).scalar()

print(f"Filas en S3 (metadata/): {len(claves_metadata)}")
print(f"Filas en RDS (pipeline_metadata): {total_rds}")

muestra = pd.read_sql("SELECT * FROM pipeline_metadata ORDER BY registered_at DESC LIMIT 10", engine)
muestra

Filas en S3 (metadata/): 117
Filas en RDS (pipeline_metadata): 117


,id,s3_bucket,s3_key,taxi_type,file_year,file_month,file_size_bytes,num_rows,registered_by,registered_at
0,103,xideralaws-curso-proyecto-alan,raw_data/2026/02/green_tripdata_2026-02.parquet,green,2026,2,920753,None,backfill-taxi-metadata,2026-09-17 08:48:58
1,109,xideralaws-curso-proyecto-alan,raw_data/2026/04/fhv_tripdata_2026-04.parquet,fhv,2026,4,23176841,None,backfill-taxi-metadata,2026-09-17 08:48:58
2,110,xideralaws-curso-proyecto-alan,raw_data/2026/04/fhvhv_tripdata_2026-04.parquet,fhvhv,2026,4,509865567,None,backfill-taxi-metadata,2026-09-17 08:48:58
3,104,xideralaws-curso-proyecto-alan,raw_data/2026/02/yellow_tripdata_2026-02.parquet,yellow,2026,2,58683353,None,backfill-taxi-metadata,2026-09-17 08:48:58
4,107,xideralaws-curso-proyecto-alan,raw_data/2026/03/green_tripdata_2026-03.parquet,green,2026,3,1082530,None,backfill-taxi-metadata,2026-09-17 08:48:58
5,117,xideralaws-curso-proyecto-alan,raw_data/2026/06/green_tripdata_2026-06.parquet,green,2026,6,951674,None,backfill-taxi-metadata,2026-09-17 08:48:58
6,106,xideralaws-curso-proyecto-alan,raw_data/2026/03/fhvhv_tripdata_2026-03.parquet,fhvhv,2026,3,532615621,None,backfill-taxi-metadata,2026-09-17 08:48:58
7,112,xideralaws-curso-proyecto-alan,raw_data/2026/04/yellow_tripdata_2026-04.parquet,yellow,2026,4,64818115,None,backfill-taxi-metadata,2026-09-17 08:48:58
8,105,xideralaws-curso-proyecto-alan,raw_data/2026/03/fhv_tripdata_2026-03.parquet,fhv,2026,3,25607058,None,backfill-taxi-metadata,2026-09-17 08:48:58
9,111,xideralaws-curso-proyecto-alan,raw_data/2026/04/green_tripdata_2026-04.parquet,green,2026,4,1075896,None,backfill-taxi-metadata,2026-09-17 08:48:58


### Cerrar la conexión

In [6]:
engine.dispose()
print("Conexion a RDS cerrada")

Conexion a RDS cerrada
